# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
! pip install -q schedule pytest requests beautifulsoup4


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: /Users/owner/.local/share/mise/installs/python/3.13.15/bin/python3.13 -m pip install --upgrade pip


In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup

/Users/owner/Desktop/dz3/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
import requests
from bs4 import BeautifulSoup


def get_book_data(book_url: str) -> dict:
    """
    Парсит инфурмацию о книге по ссылке.

    book_url (str): ссылка на книгу
    return (dict): словарь с данными о книге
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    otvet = requests.get(book_url)
    otvet.encoding = 'utf-8'
    sup = BeautifulSoup(otvet.text, 'html.parser')

    nazvanie = sup.find('h1').text

    glavnyi_blok = sup.find('div', class_='product_main')
    cena = glavnyi_blok.find('p', class_='price_color').text

    reiting_teg = glavnyi_blok.find('p', class_='star-rating')
    reiting = reiting_teg['class'][1]

    nalichie = glavnyi_blok.find('p', class_='instock').text.strip()

    opisanie_zagolovok = sup.find('div', id='product_description')
    if opisanie_zagolovok:
        opisanie = opisanie_zagolovok.find_next_sibling('p').text
    else:
        opisanie = ''

    dannie = {
        'Название': nazvanie,
        'Цена': cena,
        'Рейтинг': reiting,
        'Наличие': nalichie,
        'Описание': opisanie
    }

    tablica = sup.find('table', class_='table-striped')
    for stroka in tablica.find_all('tr'):
        klyuch = stroka.find('th').text
        znachenie = stroka.find('td').text
        dannie[klyuch] = znachenie

    return dannie
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [4]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'Название': 'A Light in the Attic',
 'Цена': '£51.77',
 'Рейтинг': 'Three',
 'Наличие': 'In stock (22 available)',
 'Описание': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [5]:
import os


def scrape_books(is_save: bool = False) -> list:
    """
    Парсит все книги из каталога.

    is_save (bool): нужно ли сохранять в файл
    return (list): это список словарей с книгами
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    vse_knigi = []
    nomer_stranicy = 1

    while True:
        ssilka_stranicy = f'http://books.toscrape.com/catalogue/page-{nomer_stranicy}.html'
        otvet = requests.get(ssilka_stranicy)
        if otvet.status_code != 200:
            break

        sup = BeautifulSoup(otvet.text, 'html.parser')
        kartochki = sup.find_all('article', class_='product_pod')

        for kartochka in kartochki:
            hvost = kartochka.find('h3').find('a')['href']
            ssilka_knigi = 'http://books.toscrape.com/catalogue/' + hvost
            kniga = get_book_data(ssilka_knigi)
            vse_knigi.append(kniga)

        nomer_stranicy += 1

    if is_save:
        os.makedirs('../artifacts', exist_ok=True)
        with open('../artifacts/books_data.txt', 'w', encoding='utf-8') as fail:
            for kniga in vse_knigi:
                fail.write(str(kniga) + '\n')

    return vse_knigi
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [6]:
# Проверка работоспособности функции
res = scrape_books(is_save=True) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

<class 'list'> 1000


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [7]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
import os
import shutil


def zapusk_sbor():
    scrape_books(is_save=True)
    os.makedirs('../artifacts', exist_ok=True)
    if os.path.exists('books_data.txt'):
        shutil.move('books_data.txt', '../artifacts/books_data.txt')


schedule.clear()
schedule.every().day.at("03:35").do(zapusk_sbor)

while True:
    schedule.run_pending()
    time.sleep(1)
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

KeyboardInterrupt: 

## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `../tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [8]:
! uv pip install -q requests beautifulsoup4 pytest
! PYTHONPATH=.. ../.venv/bin/pytest -W ignore ../tests/test_scraper.py

============================= test session starts ==============================
platform darwin -- Python 3.9.6, pytest-8.4.2, pluggy-1.6.0
rootdir: /Users/owner/Desktop/dz3
configfile: pyproject.toml
collected 3 items                                                              

../tests/test_scraper.py ...                                             [100%]

============================== 3 passed in 1.39s ===============================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `../scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `../artifacts` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `../tests` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `../scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `../artifacts` с результатом сбора данных (`.txt` файл).
* Папка `../tests` с тестами на `pytest`.
* Папка `` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: было задано сделать парсер книжек и записыватель. все получилось `

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: ничего сразу не работал. все ломалось. пришлось обратиться к иишке за объяснениями. в итоге, в венве все заработало.`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: уточнял почему у меня не работает pip. еще он мне писал ридми - точнее переписывал в мд форматировани, но и я потом половину вырезал из ридми, потому что он получлся супер иишный. еще я у иишки уточнял что надо в гитигнор добавить. еще confest.py - его совет.`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: в целом, я уже все узнал у иишки. так что теперь все понятно`

**5. Самооценка по критериям задания**:

`Ответ: по критериям - все сделал. все сработало.`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: как в прошлый раз - понравилась практика.`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.
>
>
> у меня на ноуте я делал это с venv. если вы тоже, в консоли венва вствьте это .venv/bin/python -m ensurepip

